<a href="https://colab.research.google.com/github/Harsh-s7-hub/GENAI_ASSIGNMENTS/blob/main/GenAI_FinalAssignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install -q peft accelerate bitsandbytes evaluate rouge_score

!pip install -q -U transformers
!pip install -q graphviz
print('All packages installed')
print('Restart runtime now if this is your first run!')

All packages installed
Restart runtime now if this is your first run!


In [ ]:
import json, copy, random
import numpy as np
import matplotlib.pyplot as plt

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoConfig,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Using device: cuda
   GPU: Tesla T4
   VRAM: 15.6 GB


In [ ]:

with open("university_qa_dataset.json", "r") as f:
    raw_data = json.load(f)

random.seed(42)
random.shuffle(raw_data)

print(f"Dataset loaded: {len(raw_data)} records")
print(f"   Fields: {list(raw_data[0].keys())}")
print(f"\nSample record:")
print(f"  INPUT : {raw_data[0]['input']}")
print(f"  OUTPUT: {raw_data[0]['output']}")

Dataset loaded: 1000 records
   Fields: ['input', 'output']

Sample record:
  INPUT : Can you explain a student mentor?
  OUTPUT: A student mentor is typically a senior student or faculty member assigned to guide a junior student through academic and personal challenges.


In [ ]:

train_raw = raw_data[:900]
test_raw  = raw_data[900:]

print(f"Train size: {len(train_raw)} | Test size: {len(test_raw)}")


def format_record(example):
    return {
        "text": f"Question: {example['input']}\nAnswer: {example['output']}"
    }

train_dataset_raw = Dataset.from_list(train_raw).map(format_record)
test_dataset_raw  = Dataset.from_list(test_raw)

print(f"\nFormatted sample:")
print(train_dataset_raw[0]['text'])

Train size: 900 | Test size: 100


Map:   0%|          | 0/900 [00:00<?, ? examples/s]


Formatted sample:
Question: Can you explain a student mentor?
Answer: A student mentor is typically a senior student or faculty member assigned to guide a junior student through academic and personal challenges.


In [ ]:
model_name = "microsoft/phi-2"


tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)
model.config.pad_token_id = tokenizer.pad_token_id


base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)
base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {model_name}")
print(f"   Total parameters: {total_params:,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

Model loaded: microsoft/phi-2
   Total parameters: 2,779,683,840


In [ ]:
MAX_LEN = 128

def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_train = train_dataset_raw.map(tokenize, batched=False)
tokenized_train = tokenized_train.remove_columns(
    [c for c in tokenized_train.column_names if c not in ["input_ids", "attention_mask", "labels"]]
)
tokenized_train.set_format("torch")

print(f"Tokenization done")
print(f"   Columns: {tokenized_train.column_names}")
print(f"   Sequence length: {MAX_LEN} tokens")

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Tokenization done
   Columns: ['input_ids', 'attention_mask', 'labels']
   Sequence length: 128 tokens


In [ ]:
lora_config = LoraConfig(
    r=8,                               # rank — higher = more capacity, slower
    lora_alpha=16,                     # scaling factor (alpha/r = 2.0)
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())

print(f"LoRA applied")
print(f"   Trainable parameters : {trainable:,}  ({100*trainable/total:.3f}%)")
print(f"   Frozen parameters    : {total-trainable:,}")
print(f"   Total parameters     : {total:,}")

LoRA applied
   Trainable parameters : 2,621,440  (0.094%)
   Frozen parameters    : 2,779,683,840
   Total parameters     : 2,782,305,280


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    fp16=(device == "cuda"),
    logging_steps=10,
    save_strategy="no",
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    remove_unused_columns=False,
    report_to="none",
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print("Starting training...")
train_result = trainer.train()

print("\nTraining complete!")
print(f"   Final loss  : {train_result.training_loss:.4f}")
print(f"   Total steps : {train_result.global_step}")
print(f"   Runtime     : {train_result.metrics['train_runtime']:.0f} seconds")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting training...


Step,Training Loss
10,2.543493
20,2.377246
30,1.960374
40,1.766856
50,1.705542
60,1.618602
70,1.508797
80,1.450061
90,1.495949
100,1.396782


---
## Step 9 — Training Loss Curve

In [ ]:
loss_log = [(e['step'], e['loss']) for e in trainer.state.log_history if 'loss' in e]

if loss_log:
    steps, losses = zip(*loss_log)
    plt.figure(figsize=(9, 4))
    plt.plot(steps, losses, marker='o', color='steelblue', linewidth=2, markersize=4)
    plt.fill_between(steps, losses, alpha=0.1, color='steelblue')
    plt.xlabel('Training Step', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title('Training Loss Curve — LoRA Fine-Tuning (Phi-2)', fontsize=13, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('training_loss.png', dpi=150)
    plt.show()
    print(f"Loss: {losses[0]:.4f} → {losses[-1]:.4f}  (reduced by {losses[0]-losses[-1]:.4f})")
else:
    print("No loss log found")

---
## Step 10 — Save Fine-Tuned Model

In [ ]:
model.save_pretrained("fine_tuned_model")
tokenizer.save_pretrained("fine_tuned_model")
print("Model saved to ./fine_tuned_model/")
print("   (Only LoRA adapter weights are saved — lightweight!)")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
model.save_pretrained("/content/drive/MyDrive/fine_tuned_model")
tokenizer.save_pretrained("/content/drive/MyDrive/fine_tuned_model")

---
## 💬 Step 11 — Before vs After: Qualitative Output Comparison
Both models get the same questions. Base model = no fine-tuning. Fine-tuned = LoRA trained above.

In [ ]:
def generate_answer(mdl, question):
    """Prompt engineering + output cleaning"""
    prompt = (
        "You are a university academic assistant. "
        "Answer the following question clearly and concisely.\n\n"
        f"Question: {question}\nAnswer:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = mdl.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id
        )
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)

    return decoded.split("Answer:")[-1].strip()



sample_questions = [d['input'] for d in test_raw[:4]]

model.eval()
for i, q in enumerate(sample_questions, 1):
    print(f"\n{'='*65}")
    print(f"Q{i}: {q}")
    print(f"{'='*65}")
    print("BEFORE (Base Phi-2):")
    print(" ", generate_answer(base_model, q))
    print("\nAFTER  (Fine-Tuned LoRA Phi-2):")
    print(" ", generate_answer(model, q))

---
## Step 12 — Quantitative Evaluation (BLEU + ROUGE + Accuracy)
Evaluate on 100 held-out test samples.

In [ ]:
bleu_metric  = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")

def clean_text(text):
    return text.split("Answer:")[-1].strip()

def get_predictions(mdl, samples):
    """Run inference + output cleaning on a list of {input, output} dicts"""
    preds, refs = [], []
    mdl.eval()
    for item in samples:
        prompt = f"Question: {item['input']}\nAnswer:"
        inputs = tokenizer(prompt, return_tensors="pt",
                           truncation=True, max_length=MAX_LEN).to(device)
        with torch.no_grad():
            out = mdl.generate(
                **inputs,
                max_new_tokens=60,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        pred = tokenizer.decode(out[0], skip_special_tokens=True)
        pred = clean_text(pred)
        preds.append(pred)
        refs.append(item['output'])
    return preds, refs


# Use 20 samples from test set (fast but representative)
eval_samples = test_raw[:20]

print("Evaluating base model...")
base_preds, refs = get_predictions(base_model, eval_samples)

print("Evaluating fine-tuned model...")
ft_preds, _      = get_predictions(model, eval_samples)

print("✅ Predictions generated")

---
## Step 13 — Compute All Metrics

In [ ]:
# BLEU
bleu_base = bleu_metric.compute(predictions=base_preds, references=[[r] for r in refs])
bleu_ft   = bleu_metric.compute(predictions=ft_preds,   references=[[r] for r in refs])

# ROUGE
rouge_base = rouge_metric.compute(predictions=base_preds, references=refs)
rouge_ft   = rouge_metric.compute(predictions=ft_preds,   references=refs)

# Word-overlap Accuracy (≥50% reference words match)
def word_accuracy(preds, refs):
    correct = 0
    for p, r in zip(preds, refs):
        ref_words = r.lower().split()
        matches   = sum(1 for w in ref_words if w in p.lower())
        if matches >= max(1, len(ref_words) // 2):
            correct += 1
    return correct / len(refs)

acc_base = word_accuracy(base_preds, refs)
acc_ft   = word_accuracy(ft_preds,   refs)

# ── Results table ──────────────────────────────────────────────
print("\n" + "="*60)
print(f"{'Metric':<18} {'Base':>10} {'Fine-Tuned':>12} {'Change':>10}")
print("="*60)

rows = [
    ("BLEU",     bleu_base['bleu'],          bleu_ft['bleu']),
    ("ROUGE-1",  float(rouge_base['rouge1']), float(rouge_ft['rouge1'])),
    ("ROUGE-2",  float(rouge_base['rouge2']), float(rouge_ft['rouge2'])),
    ("ROUGE-L",  float(rouge_base['rougeL']), float(rouge_ft['rougeL'])),
    ("Accuracy", acc_base,                    acc_ft),
]

for name, b, f in rows:
    delta = f - b
    sign  = "↑" if delta > 0 else ("↓" if delta < 0 else "=")
    print(f"{name:<18} {b:>10.4f} {f:>12.4f}   {sign} {abs(delta):.4f}")

print("="*60)

---
##  Step 14 — Grouped Bar Chart Comparison

In [ ]:
labels      = ["BLEU", "ROUGE-1", "ROUGE-2", "ROUGE-L", "Accuracy"]
base_scores = [rows[i][1] for i in range(5)]
ft_scores   = [rows[i][2] for i in range(5)]

x     = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))

b1 = ax.bar(x - width/2, base_scores, width,
            label="Base Model (Phi-2)",
            color="#4C72B0", alpha=0.88, edgecolor="black", linewidth=0.5)
b2 = ax.bar(x + width/2, ft_scores, width,
            label="Fine-Tuned (LoRA)",
            color="#55A868", alpha=0.88, edgecolor="black", linewidth=0.5)

for bar in b1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
            f"{h:.3f}", ha="center", va="bottom", fontsize=8.5)
for bar in b2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
            f"{h:.3f}", ha="center", va="bottom", fontsize=8.5, color="darkgreen", fontweight="bold")

ax.set_xlabel("Evaluation Metric", fontsize=12)
ax.set_ylabel("Score (0–1)", fontsize=12)
ax.set_title("Base Model vs Fine-Tuned Model — All Metrics",
             fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0, max(max(base_scores), max(ft_scores)) * 1.3)
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig("metrics_comparison.png", dpi=150)
plt.show()
print("Chart saved as metrics_comparison.png")

---
## Step 16 — Interactive Chat with Fine-Tuned Model

In [ ]:
print("University Assistant — Fine-Tuned Phi-2")
print("Type 'exit' to stop.\n")

model.eval()
while True:
    q = input("Ask: ").strip()
    if q.lower() in ("exit", "quit", ""):
        print("Goodbye!")
        break
    ans = generate_answer(model, q)
    print(f"Answer: {ans}\n")